# 110. Audio Processing: Working with Audio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/13-multi-modal/110_audio_processing.ipynb)

**Category:** 13 - Multi-Modal Techniques  
**Technique #:** 110  
**Difficulty:** Intermediate

## 📖 Description

Audio Processing with LLMs involves transcribing speech, analyzing audio content, and extracting insights from spoken language. Modern models can handle transcription, sentiment analysis, speaker identification, and content summarization from audio files.

### When to Use:
- Transcribing meetings and interviews
- Analyzing customer service calls
- Creating subtitles and captions
- Podcast and content analysis
- Voice command processing

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                   AUDIO PROCESSING FLOW                      │
└─────────────────────────────────────────────────────────────┘

    ┌──────────────┐         ┌──────────────┐         ┌──────────────┐
    │   Audio      │────────▶│   Speech-to  │────────▶│   Text       │
    │   Input      │         │   Text (STT) │         │   Transcript │
    └──────────────┘         └──────────────┘         └──────┬───────┘
         (mp3, wav, m4a)                                    │
                                                             ▼
                                                      ┌──────────────┐
                                                      │   LLM        │
                                                      │   Analysis   │
                                                      └──────┬───────┘
                                                             │
                                    ┌────────────────────────┼────────────────────────┐
                                    ▼                        ▼                        ▼
                            ┌──────────────┐         ┌──────────────┐         ┌──────────────┐
                            │  Sentiment   │         │  Summary     │         │  Insights    │
                            │  Analysis    │         │  Generation  │         │  Extraction  │
                            └──────────────┘         └──────────────┘         └──────────────┘
```

### Processing Pipeline:
1. **Audio Input**: Load audio file
2. **Speech Recognition**: Convert speech to text
3. **Text Processing**: Clean and format transcript
4. **LLM Analysis**: Extract insights, sentiment, entities
5. **Output Generation**: Structured results

## 🛠️ Setup

In [ ]:
!pip install -q openai

In [ ]:
import os
from getpass import getpass

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI()

## 💡 Basic Example

In [ ]:
def transcribe_audio(audio_file_path, model="whisper-1"):
    """Transcribe audio file to text."""
    try:
        with open(audio_file_path, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model=model,
                file=audio_file
            )
        return transcription.text
    except Exception as e:
        return f"Error: {str(e)}"

def transcribe_audio_url(audio_url, model="whisper-1"):
    """Transcribe audio from URL (downloads first)."""
    import requests
    import tempfile
    
    try:
        # Download audio file
        response = requests.get(audio_url)
        
        # Save to temp file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp:
            tmp.write(response.content)
            tmp_path = tmp.name
        
        # Transcribe
        result = transcribe_audio(tmp_path, model)
        
        # Clean up
        os.unlink(tmp_path)
        
        return result
    except Exception as e:
        return f"Error: {str(e)}"

def analyze_transcript(transcript, analysis_type="summary"):
    """Analyze transcribed text with LLM."""
    
    analysis_prompts = {
        "summary": f"Summarize the following transcript concisely:\n\n{transcript}",
        "sentiment": f"Analyze the sentiment of this transcript. Provide overall sentiment (positive/negative/neutral) and key emotional indicators:\n\n{transcript}",
        "topics": f"Extract the main topics and key points from this transcript:\n\n{transcript}",
        "action_items": f"Extract action items and tasks mentioned in this transcript:\n\n{transcript}",
        "questions": f"List all questions asked in this transcript and provide brief answers if available:\n\n{transcript}"
    }
    
    prompt = analysis_prompts.get(analysis_type, analysis_prompts["summary"])
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are an expert at analyzing transcripts and extracting insights."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=1000
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Example with sample transcript
sample_transcript = """
Welcome to our quarterly review meeting. Today we'll discuss our progress on the new product launch. 
First, Sarah will present the marketing strategy. Then we'll hear from Tom about development timelines. 
Our main goals for Q3 are increasing user engagement by 20% and launching the mobile app. 
Questions? Yes, John? When is the expected launch date? We're targeting mid-September. 
Any other questions? Great, let's move on to the budget discussion.
"""

print("AUDIO TRANSCRIPT ANALYSIS - EXAMPLE\n")
print("="*60 + "\n")

# Analyze the sample transcript
summary = analyze_transcript(sample_transcript, "summary")
print("SUMMARY:")
print(summary)
print("\n" + "-"*40 + "\n")

action_items = analyze_transcript(sample_transcript, "action_items")
print("ACTION ITEMS:")
print(action_items)

## 🌍 Real-World Example

In [ ]:
# Real-world: Customer service call analysis
def analyze_customer_call(transcript):
    """Comprehensive customer service call analysis."""
    
    prompt = f"""
    You are a customer service quality analyst. Analyze this call transcript:
    
    TRANSCRIPT:
    {transcript}
    
    Provide analysis in this format:
    
    ## Call Summary
    - Brief overview of the interaction
    - Issue type/category
    
    ## Sentiment Analysis
    - Customer sentiment (positive/neutral/negative)
    - Agent sentiment
    - Sentiment progression throughout call
    
    ## Key Metrics
    - Politeness indicators
    - Empathy shown
    - Problem resolution status
    
    ## Quality Assessment
    - Agent performance score (1-10)
    - What went well
    - Areas for improvement
    
    ## Recommendations
    - Follow-up actions needed
    - Training opportunities
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are an expert customer service analyst."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=1500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Sample customer service call
customer_call = """
Agent: Thank you for calling TechSupport. This is Alex. How can I help you today?
Customer: Hi, I'm having trouble with my internet connection. It's been really slow all day.
Agent: I'm sorry to hear that. Let me help you troubleshoot. Can you tell me what device you're using?
Customer: I'm on my laptop. I've tried restarting it but nothing changed.
Agent: Okay, let's check your router. Have you tried unplugging it for 30 seconds?
Customer: No, I haven't. Let me try that now.
Agent: Take your time. I'll wait here.
Customer: Okay, it's back on now. The lights are blinking.
Agent: Great! Give it a minute to fully reconnect, then try accessing a website.
Customer: It's working now! Much faster. Thank you so much!
Agent: You're welcome! Is there anything else I can help you with today?
Customer: No, that's all. Thanks again!
Agent: Have a great day!
"""

print("CUSTOMER SERVICE CALL ANALYSIS\n")
print("="*60 + "\n")

analysis = analyze_customer_call(customer_call)
print(analysis)

## ❌ Failure Case

In [ ]:
# Failure case: Poor audio quality challenges
print("AUDIO PROCESSING LIMITATIONS\n")
print("="*60 + "\n")

limitations = [
    {
        "issue": "Background noise",
        "impact": "Reduced transcription accuracy",
        "mitigation": "Use noise reduction preprocessing"
    },
    {
        "issue": "Multiple speakers/overlap",
        "impact": "Speaker diarization errors",
        "mitigation": "Use speaker diarization models"
    },
    {
        "issue": "Accents and dialects",
        "impact": "Lower accuracy for non-standard speech",
        "mitigation": "Use models trained on diverse data"
    },
    {
        "issue": "Technical jargon",
        "impact": "Misrecognition of specialized terms",
        "mitigation": "Use custom vocabulary/tuning"
    },
    {
        "issue": "Long audio files",
        "impact": "API limits, processing timeouts",
        "mitigation": "Split into segments, batch process"
    }
]

for lim in limitations:
    print(f"⚠️  {lim['issue']}")
    print(f"   Impact: {lim['impact']}")
    print(f"   Mitigation: {lim['mitigation']}\n")

print("="*60)
print("BEST PRACTICES:")
print("="*60)
print("""
1. Use high-quality audio (16kHz+ sample rate)
2. Minimize background noise
3. Speak clearly and at moderate pace
4. Use appropriate microphones
5. Pre-process audio for noise reduction
6. Split long files into manageable chunks
""")

## 📊 Benchmark Comparison

| Model/Service | English WER | Multilingual | Speaker ID | Real-time | Cost |
|---------------|-------------|--------------|------------|-----------|------|
| Whisper Large | 9.9% | ⭐⭐⭐⭐⭐ | ❌ | ❌ | Free |
| Whisper API | 9.9% | ⭐⭐⭐⭐⭐ | ❌ | ❌ | $ |
| Google STT | 11.2% | ⭐⭐⭐⭐ | ✅ | ✅ | $$ |
| Azure Speech | 10.5% | ⭐⭐⭐⭐ | ✅ | ✅ | $$ |
| AWS Transcribe | 11.8% | ⭐⭐⭐ | ✅ | ✅ | $$ |
| AssemblyAI | 10.1% | ⭐⭐⭐⭐ | ✅ | ✅ | $$ |

*WER = Word Error Rate (lower is better)

### Recommendations:
- **Whisper**: Best accuracy, free (self-hosted)
- **Google/Azure**: Enterprise features, speaker ID
- **AssemblyAI**: Great API, built-in features

## 🎮 Interactive Playground

In [ ]:
def audio_analysis_playground():
    """Interactive audio analysis playground."""
    print("\n" + "="*60)
    print("AUDIO PROCESSING PLAYGROUND")
    print("="*60 + "\n")
    
    print("Enter a transcript to analyze (or use sample):\n")
    user_input = input("Paste transcript (or press Enter for sample): ").strip()
    
    if not user_input:
        transcript = """
        Welcome to the product demo. Today I'll show you our new AI features. 
        First, let's look at the dashboard. You can see real-time analytics here. 
        The system automatically detects anomalies and alerts your team. 
        Any questions? Yes, how much does it cost? Pricing starts at $99 per month. 
        We also offer enterprise plans with custom features.
        """
        print("\nUsing sample transcript.\n")
    else:
        transcript = user_input
    
    print("\nSelect analysis type:")
    print("1. Summary")
    print("2. Sentiment Analysis")
    print("3. Topic Extraction")
    print("4. Action Items")
    print("5. Questions & Answers")
    
    analysis_choice = input("Enter choice (1-5): ").strip()
    
    analysis_types = {
        "1": "summary",
        "2": "sentiment",
        "3": "topics",
        "4": "action_items",
        "5": "questions"
    }
    
    selected_type = analysis_types.get(analysis_choice, "summary")
    
    print(f"\nPerforming {selected_type} analysis...\n")
    
    result = analyze_transcript(transcript, selected_type)
    
    print("="*60)
    print("ANALYSIS RESULT:")
    print("="*60)
    print(result)

audio_analysis_playground()

## 💡 Tips & Tricks

### Transcription Optimization:
1. **Audio format**: Use MP3 or WAV (16kHz, mono)
2. **Chunk size**: Split files >25MB
3. **Language**: Specify language for better accuracy
4. **Prompt**: Use context prompts for domain terms

### Analysis Enhancement:
- **Timestamps**: Request timestamped transcripts
- **Speaker labels**: Enable diarization when available
- **Confidence scores**: Filter low-confidence segments
- **Custom vocabulary**: Add domain-specific terms

### Common Use Cases:
- Meeting summaries with action items
- Call center quality monitoring
- Podcast content indexing
- Interview transcription and analysis
- Lecture notes generation

## 📚 References

1. [OpenAI Whisper](https://platform.openai.com/docs/guides/speech-to-text)
2. [Google Cloud Speech-to-Text](https://cloud.google.com/speech-to-text)
3. [Azure Speech Services](https://azure.microsoft.com/en-us/services/cognitive-services/speech-services/)
4. [AWS Transcribe](https://aws.amazon.com/transcribe/)
5. [AssemblyAI Documentation](https://www.assemblyai.com/docs/)